# Coordenação expansiva vs. coordenação concentradora: cláusula de desempenho e dispersão do NECr em 2022

O NECr sobe de forma acentuada de 2018 para 2022. A leitura direta -- "os partidos coordenam
menos em 2022" -- é apressada: o FEFC cresceu ~144% em termos nominais entre as duas eleições,
e o aumento do NECr pode refletir apenas a **margem extensiva** (mais candidatos recebendo
*algum* recurso) sob abundância orçamentária, não uma queda na concentração dos recursos
*estrategicamente relevantes* (Seção 13 de `3_necr.ipynb` já decompõe essa hipótese).

Este notebook testa uma hipótese complementar, de natureza institucional: a Emenda
Constitucional 97/2017 instituiu uma cláusula de desempenho escalonada -- para manter acesso
pleno ao Fundo Partidário e ao tempo de propaganda, o partido precisa obter, na eleição para
a Câmara dos Deputados, **1,5% dos votos válidos** (distribuídos em ao menos 1/3 das UF, com
mínimo de 1% em cada) **ou eleger 9 Deputados Federais** em 2018; o patamar sobe para **2% dos
votos ou 11 Deputados** em 2022. Esse calendário já estava fixado em 2017 -- os partidos
sabiam, ao planejar a campanha de 2022, se o desempenho de 2018 os deixava confortavelmente
acima do novo corte, próximos dele, ou abaixo.

**Hipótese revisada**: em 2018, a coordenação intrapartidária aparece como seleção de um
núcleo restrito de candidaturas viáveis, aproximando o NECr do patamar $M_p+1$ (Cox 1997;
Crisp et al. 2007). Em 2022, o fim das coligações proporcionais, o endurecimento da cláusula
de desempenho e a ampliação do FEFC alteraram os incentivos: como o acesso futuro a recursos
depende da votação e das cadeiras conquistadas *nacionalmente*, e não apenas da eleição de
cada candidato individualmente, partidos sob risco de não atingir o novo corte (ou perto dele)
têm razão para financiar um conjunto mais amplo de candidaturas -- inclusive de baixa
viabilidade individual -- que contribuam para a votação agregada da legenda, sua distribuição
territorial e sua sobrevivência institucional. Se essa hipótese está correta, o aumento do
NECr em 2022 deve ser **maior entre os partidos vulneráveis/marginais à cláusula de
desempenho** do que entre os partidos já seguros -- um padrão de diferenças-em-diferenças
(DiD).

**Desenho**:
1. Classificar partidos por vulnerabilidade à cláusula de desempenho, usando o desempenho
   real de 2018 (votos % nacionais e Deputados Federais eleitos) frente ao patamar de 2022
   (2% / 11 deputados) -- essa informação já era conhecida pelos partidos ao planejar 2022.
2. Construir métricas de esforço extensivo por lista (partido × UF): `n_fundados`,
   `prop_fundados`, `share_top_(Mp+1)`, `tail_share_(Mp+1)` e a proporção de votos/recursos
   fora do núcleo top $M_p+1$.
3. Testar, via modelo de diferenças-em-diferenças (`Ano2022 × Vulnerabilidade`), se a
   dispersão cresceu mais nos partidos sob ameaça.
4. Diferenciar partidos por porte (pequeno/médio/grande, com base na bancada nacional de
   2018), já que o mecanismo esperado é distinto entre sobrevivência (partidos pequenos/
   vulneráveis) e maximização de bancada futura (partidos grandes).

**Limitação declarada**: a classificação de vulnerabilidade usa apenas a sigla partidária
consolidada pela harmonização canônica do projeto (`PCdoB→PC do B`, `PP**→PP`,
`SD→SOLIDARIEDADE`, `DEM→UNIÃO`, `PR→PL`, `PRB→REPUBLICANOS`) e o critério simplificado de
votos % nacionais / Deputados Federais eleitos nacionalmente -- sem impor o requisito legal
completo de distribuição em ao menos 1/3 das UF com mínimo de 1% em cada. Federações
partidárias formadas especificamente para 2022 (que preservam a sigla de origem, ao contrário
de fusões como DEM+PSL→UNIÃO) não são capturadas nesta classificação; isso pode atenuar a
vulnerabilidade medida de partidos que se federaram por essa exata razão.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src" / "2_gold"))

from cap3_cs_features import gerar_features, carregar_rrd

pd.set_option("display.width", 140)

rrd = gerar_features(carregar_rrd())
rrd = rrd[rrd.ano_eleicao.isin([2018, 2022])].copy()
print(f"rrd (2018+2022): {len(rrd):,} candidatos")

rrd (2018+2022): 17,305 candidatos


## 1. Magnitude partidária ($M_p$)

Mesma construção de `3_necr.ipynb`: bancada estadual prévia (véspera das convenções),
harmonizada pela sigla canônica do projeto.

In [2]:
HARMONIZACAO = {
    "PP**": "PP",
    "PCdoB": "PC do B",
    "PTdoB": "PT do B",
    "SD": "SOLIDARIEDADE",
    "DEM": "UNIÃO",
    "PR": "PL",
    "PRB": "REPUBLICANOS",
}

bancada = pd.read_csv(ROOT / "data" / "processed" / "bancada_partido_uf.csv")
bancada["sg_partido"] = bancada["sg_partido"].replace(HARMONIZACAO)

bancada_uf = (
    bancada[bancada.ano_eleicao.isin([2018, 2022])]
    [["ano_eleicao", "sg_uf", "sg_partido", "n_deputados"]]
    .rename(columns={"n_deputados": "Mp"})
)

rrd = rrd.merge(bancada_uf, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left")
rrd["Mp"] = rrd["Mp"].fillna(0).astype(int)

n_listas = rrd.drop_duplicates(["ano_eleicao", "sg_uf", "sg_partido"]).shape[0]
print(f"Listas partido x UF x ano (2018+2022): {n_listas:,}")

Listas partido x UF x ano (2018+2022): 1,570


## 2. Classificação de vulnerabilidade à cláusula de desempenho

Usa o desempenho **real de 2018** (votos válidos nacionais para Deputado Federal e Deputados
Federais eleitos, ambos já com a sigla harmonizada para o padrão de 2022) frente ao patamar
que passaria a valer em **2022** (2% dos votos válidos nacionais OU 11 Deputados Federais
eleitos, cada critério bastando isoladamente por lei).

`score = max(votos_pct_2018 / 2, deputados_2018 / 11)`. Um partido com `score >= 1.5` tem
folga confortável acima do novo corte (**Seguro**); `0.5 <= score < 1.5` está na faixa de
risco moderado (**Marginal**); `score < 0.5` está claramente abaixo do patamar de 2022
(**Vulnerável**). Os limiares são arbitrários mas simétricos em torno do corte legal
(`score = 1`); a Seção 6 checa a robustez a limiares alternativos.

In [3]:
resultados = pd.read_parquet(ROOT / "data" / "processed" / "resultados.parquet")
dep_fed_2018 = resultados[
    (resultados.ano_eleicao == 2018)
    & (resultados.ds_cargo == "Deputado Federal")
    & (resultados.nr_turno == 1)
].copy()
dep_fed_2018["sg_partido"] = dep_fed_2018["sg_partido"].replace(HARMONIZACAO)

total_votos_2018 = dep_fed_2018["qt_votos_nominais"].sum()
votos_pct_2018 = (
    dep_fed_2018.groupby("sg_partido")["qt_votos_nominais"].sum() / total_votos_2018 * 100
)

deputados_2018 = (
    bancada[bancada.ano_eleicao == 2022].groupby("sg_partido")["n_deputados"].sum()
)

partidos_2022 = sorted(rrd.loc[rrd.ano_eleicao == 2022, "sg_partido"].unique())

vuln = pd.DataFrame(index=partidos_2022)
vuln.index.name = "sg_partido"
vuln["votos_pct_2018"] = votos_pct_2018.reindex(partidos_2022).fillna(0)
vuln["deputados_2018"] = deputados_2018.reindex(partidos_2022).fillna(0)
vuln["score_clausula"] = np.maximum(vuln["votos_pct_2018"] / 2.0, vuln["deputados_2018"] / 11.0)


def classificar(score):
    if score >= 1.5:
        return "Seguro"
    if score >= 0.5:
        return "Marginal"
    return "Vulneravel"


vuln["vulnerabilidade"] = vuln["score_clausula"].apply(classificar)
vuln = vuln.reset_index().sort_values("score_clausula", ascending=False)

print("Classificação de vulnerabilidade à cláusula de desempenho (com base em 2018)\n")
print(vuln.round(2).to_string(index=False))
print()
print(vuln["vulnerabilidade"].value_counts().to_string())

Classificação de vulnerabilidade à cláusula de desempenho (com base em 2018)

   sg_partido  votos_pct_2018  deputados_2018  score_clausula vulnerabilidade
           PL            5.54            77.0            7.00          Seguro
           PP            5.45            57.0            5.18          Seguro
           PT            9.60            56.0            5.09          Seguro
        UNIÃO            4.80            53.0            4.82          Seguro
          PSD            6.06            47.0            4.27          Seguro
 REPUBLICANOS            5.29            43.0            3.91          Seguro
          MDB            5.60            37.0            3.36          Seguro
         PSDB            5.92            21.0            2.96          Seguro
          PSB            5.72            24.0            2.86          Seguro
          PDT            4.32            19.0            2.16          Seguro
         PSOL            2.92             8.0            1.46   

## 3. Métricas de esforço extensivo por lista (partido × UF)

Para cada lista com recursos de partido distribuídos, em 2018 e 2022:

- `n_fundados` / `prop_fundados`: quantos e qual proporção dos candidatos da lista recebem
  algum recurso do partido;
- `share_top_mpp1`: participação dos $M_p+1$ candidatos mais financiados no total de recursos
  da lista;
- `tail_share_mpp1 = 1 - share_top_mpp1`: participação de recursos fora do núcleo mais
  financiado -- a métrica central do "esforço expansivo";
- `prop_votos_fora_top_mpp1`: proporção da votação nominal da legenda que veio de candidatos
  fora do núcleo top $M_p+1$ em recursos -- testa se a dispersão tem contrapartida eleitoral;
- `NECr`, `NECr_sobre_Mpp1`: como em `3_necr.ipynb`, para comparação direta.

In [4]:
def metricas_lista(g):
    n_cands = len(g)
    valores = g["vr_receita_recursos_partidos"].to_numpy(dtype=float)
    votos = g["qt_votos_nominais"].to_numpy(dtype=float)
    props = g["prop_vr_receita_candidato"].to_numpy(dtype=float)
    mp = int(g["Mp"].iloc[0])
    k = mp + 1

    n_fundados = int((valores > 0).sum())
    total_rec = valores.sum()
    total_votos = votos.sum()

    ordem = np.argsort(-valores)
    idx_top = ordem[:k]
    idx_resto = ordem[k:]

    rec_top = valores[idx_top].sum()
    votos_fora = votos[idx_resto].sum() if len(idx_resto) else 0.0

    sum_sq = (props ** 2).sum()
    necr = (1 / sum_sq) if sum_sq > 0 else np.nan

    return pd.Series({
        "Mp": mp,
        "dm_cat": g["dm_cat"].iloc[0],
        "n_cands": n_cands,
        "n_fundados": n_fundados,
        "prop_fundados": n_fundados / n_cands if n_cands > 0 else np.nan,
        "total_rec": total_rec,
        "total_votos": total_votos,
        "share_top_mpp1": (rec_top / total_rec) if total_rec > 0 else np.nan,
        "prop_votos_fora_top_mpp1": (votos_fora / total_votos) if total_votos > 0 else np.nan,
        "NECr": necr,
        "NECr_sobre_Mpp1": (necr / (mp + 1)) if pd.notna(necr) else np.nan,
    })


listas = (
    rrd.groupby(["ano_eleicao", "sg_uf", "sg_partido"], observed=True)
    .apply(metricas_lista, include_groups=False)
    .reset_index()
)
listas = listas[listas["total_rec"] > 0].copy()
listas["tail_share_mpp1"] = 1 - listas["share_top_mpp1"]

print(f"Listas com recursos (2018+2022): {len(listas):,}")
print(listas.groupby("ano_eleicao").size().to_string())

Listas com recursos (2018+2022): 1,434
ano_eleicao
2018    786
2022    648


## 4. Painel: listas + vulnerabilidade + porte

Junta a classificação de vulnerabilidade (nível partido, constante entre 2018 e 2022-- é uma
característica pré-determinada pelo desempenho de 2018) ao painel de listas. Adiciona `porte`
(tercis de Deputados Federais eleitos em 2018, uma proxy de tamanho nacional do partido
independente do corte de vulnerabilidade) e a dummy `ano2022`.

In [5]:
listas = listas.merge(vuln[["sg_partido", "vulnerabilidade", "score_clausula", "deputados_2018"]],
                       on="sg_partido", how="left")

listas["ano2022"] = (listas["ano_eleicao"] == 2022).astype(int)
listas["vulnerabilidade"] = pd.Categorical(
    listas["vulnerabilidade"], categories=["Seguro", "Marginal", "Vulneravel"], ordered=True
)

porte_labels = ["Pequeno", "Médio", "Grande"]
listas["porte"] = pd.qcut(
    listas["deputados_2018"].rank(method="first"), 3, labels=porte_labels
)

print("Distribuição de listas por vulnerabilidade x ano\n")
print(pd.crosstab(listas["vulnerabilidade"], listas["ano_eleicao"]).to_string())
print()
print("Distribuição de listas por porte x vulnerabilidade\n")
print(pd.crosstab(listas["porte"], listas["vulnerabilidade"]).to_string())

Distribuição de listas por vulnerabilidade x ano

ano_eleicao      2018  2022
vulnerabilidade            
Seguro            185   254
Marginal          247   237
Vulneravel        123   157

Distribuição de listas por porte x vulnerabilidade

vulnerabilidade  Seguro  Marginal  Vulneravel
porte                                        
Pequeno               0       142         259
Médio                38       342          21
Grande              401         0           0


## 5. Estatísticas descritivas: dispersão por vulnerabilidade e ano

Médias das variáveis de interesse, por grupo de vulnerabilidade e ano. Sob a hipótese
revisada, espera-se que `NECr`, `NECr_sobre_Mpp1`, `n_fundados`, `prop_fundados` e
`tail_share_mpp1` cresçam mais (e `share_top_mpp1` caia mais) em 2022 entre os grupos
Marginal e Vulnerável do que entre os Seguros.

In [6]:
variaveis_dv = [
    "NECr", "NECr_sobre_Mpp1", "n_fundados", "prop_fundados",
    "share_top_mpp1", "tail_share_mpp1", "prop_votos_fora_top_mpp1",
]

desc = (
    listas.groupby(["vulnerabilidade", "ano_eleicao"], observed=True)[variaveis_dv]
    .mean()
    .round(3)
)
print(desc.to_string())

                              NECr  NECr_sobre_Mpp1  n_fundados  prop_fundados  share_top_mpp1  tail_share_mpp1  prop_votos_fora_top_mpp1
vulnerabilidade ano_eleicao                                                                                                              
Seguro          2018         3.164            1.323       7.919          0.893           0.829            0.171                     0.251
                2022         6.854            3.143      16.248          0.952           0.542            0.458                     0.457
Marginal        2018         3.087            2.741       7.296          0.842           0.719            0.281                     0.398
                2022         5.055            4.241      12.131          0.929           0.508            0.492                     0.543
Vulneravel      2018         2.627            2.616       5.171          0.752           0.710            0.290                     0.467
                2022         5.640

In [7]:
deltas = (
    listas.groupby(["vulnerabilidade", "ano_eleicao"], observed=True)[variaveis_dv]
    .mean()
    .unstack("ano_eleicao")
)
for var in variaveis_dv:
    deltas[(var, "delta_2018_2022")] = deltas[(var, 2022)] - deltas[(var, 2018)]

delta_tab = deltas[[(var, "delta_2018_2022") for var in variaveis_dv]]
delta_tab.columns = variaveis_dv
print("Variação 2018 -> 2022, por grupo de vulnerabilidade\n")
print(delta_tab.round(3).to_string())

Variação 2018 -> 2022, por grupo de vulnerabilidade

                  NECr  NECr_sobre_Mpp1  n_fundados  prop_fundados  share_top_mpp1  tail_share_mpp1  prop_votos_fora_top_mpp1
vulnerabilidade                                                                                                              
Seguro           3.690            1.820       8.329          0.060          -0.286            0.286                     0.206
Marginal         1.968            1.500       4.835          0.087          -0.211            0.211                     0.144
Vulneravel       3.014            2.845       4.938          0.145          -0.233            0.233                     0.165


## 6. Modelo de diferenças-em-diferenças

`DV ~ ano2022 * vulnerabilidade + C(dm_cat)`, com erros-padrão *cluster* por `sg_partido`
(listas do mesmo partido em UF diferentes não são independentes). Categoria de referência:
`vulnerabilidade = Seguro`. Os coeficientes de interesse são as interações
`ano2022:vulnerabilidade[Marginal]` e `ano2022:vulnerabilidade[Vulneravel]`: se positivos (ou
negativos, para `share_top_mpp1`) e significativos, indicam que a dispersão cresceu mais nos
partidos sob ameaça da cláusula do que nos partidos seguros -- consistente com a hipótese de
coordenação expansiva por sobrevivência.

In [8]:
def rodar_did(dv, df=listas):
    formula = f"{dv} ~ ano2022 * C(vulnerabilidade, Treatment(reference=\'Seguro\')) + C(dm_cat)"
    cols_modelo = [dv, "ano2022", "vulnerabilidade", "dm_cat", "sg_partido"]
    df_modelo = df.dropna(subset=cols_modelo).copy()
    modelo = smf.ols(formula, data=df_modelo).fit(
        cov_type="cluster", cov_kwds={"groups": df_modelo["sg_partido"]}
    )
    termos_interacao = [t for t in modelo.params.index if "ano2022:" in t]
    linhas = []
    for t in termos_interacao:
        linhas.append({
            "dv": dv,
            "termo": t,
            "coef": modelo.params[t],
            "se": modelo.bse[t],
            "p_valor": modelo.pvalues[t],
        })
    return pd.DataFrame(linhas), modelo


resultados_did = []
modelos = {}
for dv in variaveis_dv:
    tab, modelo = rodar_did(dv)
    resultados_did.append(tab)
    modelos[dv] = modelo

resultados_did = pd.concat(resultados_did, ignore_index=True)
resultados_did["sig"] = np.select(
    [resultados_did["p_valor"] < 0.01, resultados_did["p_valor"] < 0.05, resultados_did["p_valor"] < 0.10],
    ["***", "**", "*"],
    default="",
)
print("Coeficientes de interação Ano2022 x Vulnerabilidade (ref.: Seguro)\n")
print(resultados_did.round(4).to_string(index=False))

Coeficientes de interação Ano2022 x Vulnerabilidade (ref.: Seguro)

                      dv                                                                   termo    coef     se  p_valor sig
                    NECr   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal] -1.8108 1.1153   0.1045    
                    NECr ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel] -0.7809 1.1547   0.4988    
         NECr_sobre_Mpp1   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal] -0.3486 0.9173   0.7040    
         NECr_sobre_Mpp1 ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel]  0.9895 0.9184   0.2813    
              n_fundados   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal] -3.8977 2.7952   0.1632    
              n_fundados ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel] -3.8512 2.5955   0.1379    
           prop_fundados   ano2022:C(vulnerabilidade, Tre

In [9]:
# Detalhe do modelo para a variável central da hipótese (NECr/(Mp+1))
print(modelos["NECr_sobre_Mpp1"].summary())

                            OLS Regression Results                            
Dep. Variable:        NECr_sobre_Mpp1   R-squared:                       0.142
Model:                            OLS   Adj. R-squared:                  0.137
Method:                 Least Squares   F-statistic:                     15.41
Date:                Tue, 30 Jun 2026   Prob (F-statistic):           1.65e-08
Time:                        22:06:00   Log-Likelihood:                -3135.8
No. Observations:                1203   AIC:                             6288.
Df Residuals:                    1195   BIC:                             6328.
Df Model:                           7                                         
Covariance Type:              cluster                                         
                                                                              coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

## 7. Diferenciação por porte partidário

O mecanismo esperado difere por porte: em partidos pequenos/vulneráveis, a dispersão é
motivada por sobrevivência institucional (evitar perder acesso a recursos e tempo de TV); em
partidos grandes, por maximização de bancada e de recursos futuros. Abaixo, a variação
2018→2022 das mesmas métricas, agora cruzando porte × vulnerabilidade.

In [10]:
desc_porte = (
    listas.groupby(["porte", "vulnerabilidade", "ano_eleicao"], observed=True)[variaveis_dv]
    .mean()
    .round(3)
)
print(desc_porte.to_string())

                                      NECr  NECr_sobre_Mpp1  n_fundados  prop_fundados  share_top_mpp1  tail_share_mpp1  prop_votos_fora_top_mpp1
porte   vulnerabilidade ano_eleicao                                                                                                              
Pequeno Marginal        2018         2.517            2.113       5.590          0.836           0.762            0.238                     0.355
                        2022         3.591            3.240      10.422          0.851           0.552            0.448                     0.622
        Vulneravel      2018         2.627            2.616       5.171          0.752           0.710            0.290                     0.467
                        2022         5.083            5.062       8.904          0.895           0.504            0.496                     0.612
Médio   Seguro          2018         2.321            1.626       7.148          0.802           0.790            0.210     

In [11]:
deltas_porte = (
    listas.groupby(["porte", "vulnerabilidade", "ano_eleicao"], observed=True)[variaveis_dv]
    .mean()
    .unstack("ano_eleicao")
)
for var in variaveis_dv:
    deltas_porte[(var, "delta_2018_2022")] = deltas_porte[(var, 2022)] - deltas_porte[(var, 2018)]

delta_porte_tab = deltas_porte[[(var, "delta_2018_2022") for var in variaveis_dv]]
delta_porte_tab.columns = variaveis_dv
print("Variação 2018 -> 2022, por porte x vulnerabilidade\n")
print(delta_porte_tab.round(3).to_string())

Variação 2018 -> 2022, por porte x vulnerabilidade

                          NECr  NECr_sobre_Mpp1  n_fundados  prop_fundados  share_top_mpp1  tail_share_mpp1  prop_votos_fora_top_mpp1
porte   vulnerabilidade                                                                                                              
Pequeno Marginal         1.073            1.128       4.832          0.015          -0.210            0.210                     0.267
        Vulneravel       2.456            2.446       3.734          0.143          -0.206            0.206                     0.145
Médio   Seguro           4.556            3.537       9.397          0.185          -0.355            0.355                     0.166
        Marginal         2.247            1.580       4.680          0.113          -0.207            0.207                     0.095
        Vulneravel         NaN              NaN         NaN            NaN             NaN              NaN                       NaN
Grande  Se

**Nota de leitura da Seção 7**: a tabela de contingência porte x vulnerabilidade (Seção 4)
mostra que os dois grupos são quase deterministicamente sobrepostos -- todas as listas
Grande são Seguro, praticamente todas as Pequeno são Marginal/Vulnerável -- porque ambos
derivam do mesmo indicador de base (`deputados_2018`). Isso não é um artefato indesejado: é
consistente com o fato de que, no sistema partidário brasileiro, tamanho da bancada e
distância da cláusula de desempenho são, por construção institucional, praticamente a mesma
coisa. Mas significa que a Seção 7 tem pouco poder de identificação *independente* do
resultado da Seção 6 -- não permite isolar um efeito de "porte" que não seja também um
efeito de "vulnerabilidade". Uma checagem futura poderia usar uma proxy de porte ortogonal
à cláusula (ex.: nacionalização do voto, idade do partido).

## 8. Robustez: limiares alternativos de vulnerabilidade

Reclassifica com limiares mais e menos exigentes (`score < 0.3/0.7` vs. `score < 0.5/1.5` vs.
`score < 0.7/1.3`) e reestima o coeficiente-chave (`ano2022 x Vulneravel` sobre
`NECr_sobre_Mpp1`) em cada especificação, para checar se o resultado depende da escolha
específica dos cortes.

In [12]:
def classificar_com_limiares(score, lim_baixo, lim_alto):
    if score >= lim_alto:
        return "Seguro"
    if score >= lim_baixo:
        return "Marginal"
    return "Vulneravel"


especificacoes = {
    "limiares_originais (0.5 / 1.5)": (0.5, 1.5),
    "limiares_estreitos (0.7 / 1.3)": (0.7, 1.3),
    "limiares_largos (0.3 / 0.7)": (0.3, 0.7),
}

linhas_robustez = []
for nome, (lim_baixo, lim_alto) in especificacoes.items():
    vuln_alt = vuln.copy()
    vuln_alt["vulnerabilidade_alt"] = vuln_alt["score_clausula"].apply(
        lambda s: classificar_com_limiares(s, lim_baixo, lim_alto)
    )
    listas_alt = listas.drop(columns=["vulnerabilidade"]).merge(
        vuln_alt[["sg_partido", "vulnerabilidade_alt"]], on="sg_partido", how="left"
    )
    listas_alt = listas_alt.rename(columns={"vulnerabilidade_alt": "vulnerabilidade"})
    listas_alt["vulnerabilidade"] = pd.Categorical(
        listas_alt["vulnerabilidade"], categories=["Seguro", "Marginal", "Vulneravel"], ordered=True
    )
    tab, _ = rodar_did("NECr_sobre_Mpp1", df=listas_alt)
    tab["especificacao"] = nome
    linhas_robustez.append(tab)

robustez = pd.concat(linhas_robustez, ignore_index=True)
print("Robustez: coeficiente ano2022 x vulnerabilidade sobre NECr/(Mp+1), por especificação\n")
print(robustez[["especificacao", "termo", "coef", "se", "p_valor"]].round(4).to_string(index=False))

Robustez: coeficiente ano2022 x vulnerabilidade sobre NECr/(Mp+1), por especificação

                 especificacao                                                                   termo    coef     se  p_valor
limiares_originais (0.5 / 1.5)   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal] -0.3486 0.9173   0.7040
limiares_originais (0.5 / 1.5) ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel]  0.9895 0.9184   0.2813
limiares_estreitos (0.7 / 1.3)   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal]  1.6315 1.0709   0.1276
limiares_estreitos (0.7 / 1.3) ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel]  1.5959 1.0458   0.1270
   limiares_largos (0.3 / 0.7)   ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Marginal]  0.9707 1.2375   0.4328
   limiares_largos (0.3 / 0.7) ano2022:C(vulnerabilidade, Treatment(reference='Seguro'))[T.Vulneravel]  0.8866 1.3080   0.4979


## 9. Síntese

- A classificação de vulnerabilidade à cláusula de desempenho, construída a partir do
  desempenho real de 2018 frente ao patamar de 2022, produz três grupos balanceados (Seção
  2).
- As tabelas descritivas (Seção 5) e o modelo de diferenças-em-diferenças (Seção 6) permitem
  avaliar diretamente se a dispersão de recursos em 2022 foi maior nos partidos sob ameaça
  institucional (Marginal/Vulnerável) do que nos partidos já seguros -- o teste central da
  hipótese de coordenação expansiva por sobrevivência.
- A Seção 7 diferencia esse padrão por porte partidário, permitindo distinguir sobrevivência
  (pequenos/vulneráveis) de maximização de bancada futura (grandes).
- A Seção 8 checa se os resultados são sensíveis à escolha específica dos limiares de
  vulnerabilidade.

**Leitura sugerida dos coeficientes**: sinais positivos em `ano2022:vulnerabilidade[T.
Marginal]` / `[T.Vulneravel]` para `NECr`, `NECr_sobre_Mpp1`, `n_fundados`, `prop_fundados`,
`tail_share_mpp1` e `prop_votos_fora_top_mpp1` -- e negativos para `share_top_mpp1` -- são
consistentes com a hipótese revisada: a coordenação não desaparece em 2022, mas muda de forma,
de concentradora para expansiva, e essa mudança é mais pronunciada exatamente nos partidos
para os quais o cálculo de sobrevivência institucional é mais relevante.